# 01 — Exploratory Data Analysis
## Deepfake Face Dataset

### Imports & configuration

In [ ]:
import hashlib
import json
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# --- Resolve project root dynamically (works on any machine) ---
def _find_project_root():
    """Walk up from notebook/script location until we find data/raw/."""
    try:
        start = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        start = os.getcwd()

    d = os.path.abspath(start)
    for _ in range(5):
        if os.path.isdir(os.path.join(d, "data", "raw")):
            return d
        d = os.path.dirname(d)
    raise FileNotFoundError(
        "Could not locate project root (looking for data/raw/). "
        "Run this notebook from the project root or the notebooks/ directory."
    )

PROJECT_ROOT = _find_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Reports dir  : {REPORTS_DIR}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

### 3.1.1 Build image inventory & count real / fake images

In [ ]:
# Walk the raw data directory and collect metadata for every image file
records = []

for root, dirs, files in os.walk(DATA_DIR):
    for fname in files:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in IMAGE_EXTENSIONS:
            continue

        path = os.path.join(root, fname)
        label = os.path.basename(root).lower()

        try:
            with Image.open(path) as img:
                width, height = img.size
                mode = img.mode
                fmt = img.format
                img.verify()

            records.append({
                "path": path,
                "filename": fname,
                "label": label,
                "extension": ext,
                "width": width,
                "height": height,
                "aspect_ratio": round(width / height, 4),
                "mode": mode,
                "format": fmt,
                "file_size_kb": round(os.path.getsize(path) / 1024, 2),
                "is_readable": True,
                "error": None
            })
        except Exception as e:
            records.append({
                "path": path,
                "filename": fname,
                "label": label,
                "extension": ext,
                "width": None,
                "height": None,
                "aspect_ratio": None,
                "mode": None,
                "format": None,
                "file_size_kb": round(os.path.getsize(path) / 1024, 2),
                "is_readable": False,
                "error": str(e)
            })

df = pd.DataFrame(records)
print(f"Total images found: {len(df):,}")
df.head()

In [ ]:
# --- Class distribution ---
counts = df["label"].value_counts()
print("Class counts:")
print(counts.to_string())
print(f"\nImbalance ratio: {counts.max() / counts.min():.2f}:1")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=["#ff6b6b", "#4ecdc4"])
ax.bar_label(bars, fmt=lambda x: f"{x:,}")
ax.set_title("Class Distribution", fontsize=13)
ax.set_ylabel("Number of Images")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

### 3.1.2 Image sizes & aspect ratios

In [ ]:
readable = df[df["is_readable"]]

# Resolution stats per class
size_stats = df.groupby("label")[["width", "height", "aspect_ratio"]].agg(
    ["count", "mean", "std", "min", "max"]
)
print("Image dimensions per class:")
display(size_stats)

# Unique resolutions in the dataset
unique_resolutions = readable.groupby(["width", "height"]).size().reset_index(name="count")
print(f"\nUnique (width, height) combinations: {len(unique_resolutions)}")
display(unique_resolutions)

In [ ]:
# Aspect-ratio histogram
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ["#ff6b6b" if l == "fake faces" else "#4ecdc4" for l in df["label"]]
axes[0].scatter(df["width"], df["height"], alpha=0.15, s=4, c=colors)
axes[0].set_title("Width vs Height")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")

axes[1].hist(readable["width"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_title("Width Distribution")
axes[1].set_xlabel("Width (px)")

axes[2].hist(readable["aspect_ratio"], bins=30, color="steelblue", edgecolor="white")
axes[2].set_title("Aspect Ratio Distribution")
axes[2].set_xlabel("Aspect Ratio (W/H)")

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "image_dimensions.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# File-size comparison between classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (lbl, clr) in zip(axes, [("fake faces", "#ff6b6b"), ("real faces", "#4ecdc4")]):
    subset = readable[readable["label"] == lbl]["file_size_kb"]
    ax.hist(subset, bins=60, color=clr, alpha=0.8, edgecolor="white")
    ax.set_title(f"File Size — {lbl}")
    ax.set_xlabel("Size (KB)")

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "file_size_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("File-size stats per class:")
display(readable.groupby("label")["file_size_kb"].describe())

### 3.1.3 File formats

In [ ]:
# Extension counts
print("File extensions:")
display(df["extension"].value_counts())

# Colour-mode counts
print("Colour modes:")
display(df["mode"].value_counts())

# Cross-tabs: label vs extension and label vs mode
print("Label vs Extension:")
display(pd.crosstab(df["label"], df["extension"]))

print("Label vs Mode:")
display(pd.crosstab(df["label"], df["mode"]))

### 3.1.4 Corrupted or unreadable images

In [ ]:
bad = df[~df["is_readable"]]
print(f"Corrupted/unreadable images: {len(bad)} / {len(df)} ({100*len(bad)/len(df):.2f}%)")

if len(bad) > 0:
    display(bad[["path", "filename", "label", "error"]])
else:
    print("No corrupted images detected.")

### 3.1.5 Duplicate & near-duplicate images

In [ ]:
def file_hash(path, algo="md5"):
    """Compute hash of file contents."""
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def pixel_hash(path):
    """Compute hash of pixel data (ignoring metadata). Useful for near-duplicate detection."""
    try:
        img = Image.open(path).convert("RGB")
        arr = np.asarray(img)
        return hashlib.md5(arr.tobytes()).hexdigest()
    except Exception:
        return None

# Exact duplicates (file-content hash)
print("Computing file-hash fingerprints...")
df["file_hash"] = [file_hash(p) for p in tqdm(df[df["is_readable"]]["path"], desc="Hashing files")]

hash_counts = df["file_hash"].value_counts()
dup_hashes = hash_counts[hash_counts > 1]
print(f"\nExact duplicate groups: {len(dup_hashes)}")
print(f"Total duplicate images (beyond first): {dup_hashes.sum() - len(dup_hashes)}")

if len(dup_hashes) > 0:
    print("\nSample duplicate groups:")
    for h in dup_hashes.head(5).index:
        display(df[df["file_hash"] == h][["filename", "label"]])
else:
    print("No exact duplicates found.")

In [ ]:
# Near-duplicate check via pixel hash
print("Computing pixel-hash fingerprints...")
df["pixel_hash"] = [pixel_hash(p) for p in tqdm(df[df["is_readable"]]["path"], desc="Hashing pixels")]

px_hash_counts = df["pixel_hash"].dropna().value_counts()
px_dup = px_hash_counts[px_hash_counts > 1]
print(f"\nNear-duplicate groups (identical pixels): {len(px_dup)}")
print(f"Total near-duplicate images (beyond first): {px_dup.sum() - len(px_dup)}")

# Cross-label duplicates (same image appearing in both classes — a label leak)
cross_px = df.dropna(subset=["pixel_hash"]).groupby("pixel_hash")["label"].nunique()
cross_dups = cross_px[cross_px > 1]
print(f"\nPixel-hash groups appearing in both classes: {len(cross_dups)}")

if len(cross_dups) > 0:
    print("WARNING: Identical pixel content found in BOTH real and fake labels!")
    for ph in cross_dups.head(5).index:
        display(df[df["pixel_hash"] == ph][["filename", "label"]])
else:
    print("No cross-class duplicates detected.")

### 3.1.6 Label leakage checks

In [ ]:
# 1. Labels from folder names
print("=== 1. Folder-name label inference ===")
folder_labels = df["label"].unique()
print(f"Unique folder-derived labels: {list(folder_labels)}")
print(f"Labels match expected (fake faces, real faces): {set(folder_labels) <= {'fake faces', 'real faces'}}")

# 2. Filename leakage — check if filenames encode the label
print("\n=== 2. Filename leakage ===")
prefixes = df["filename"].str.extract(r"^([a-zA-Z]+)")[0].value_counts()
print("Filename prefixes:")
print(prefixes.to_string())

# Check: does every 'fake_*' file belong to 'fake faces' and every 'real_*' to 'real faces'?
prefix_label = df.copy()
prefix_label["prefix"] = prefix_label["filename"].str.extract(r"^([a-zA-Z]+)")[0]
misaligned = prefix_label[~prefix_label.apply(lambda r: r["label"].startswith(r["prefix"]), axis=1)]
print(f"\nFiles whose prefix doesn't match their folder label: {len(misaligned)}")

# 3. Resolution leakage — does one class use different resolutions?
print("\n=== 3. Resolution leakage ===")
res_by_label = readable.groupby(["label", "width", "height"]).size().reset_index(name="count")
display(res_by_label)
print("All images are 256x256 — no resolution-based leakage." if len(res_by_label) == 2 else "Different resolutions between classes detected.")

# 4. Metadata leakage — check EXIF
print("\n=== 4. Metadata / EXIF leakage ===")
exif_samples = {"fake": None, "real": None}
for _, row in readable.iterrows():
    label_key = "fake" if row["label"] == "fake faces" else "real"
    if exif_samples[label_key] is None:
        try:
            exif = Image.open(row["path"]).getexif()
            exif_samples[label_key] = dict(exif) if exif else {}
        except Exception:
            exif_samples[label_key] = {}
    if all(v is not None for v in exif_samples.values()):
        break

for k, v in exif_samples.items():
    print(f"  {k}: {len(v)} EXIF tags")
    if v:
        for tag, val in list(v.items())[:5]:
            val_str = str(val)[:80]
            print(f"    {tag}: {val_str}")

# 5. Check metadata.csv for consistency with folder labels
print("\n=== 5. metadata.csv consistency ===")
meta_path = os.path.join(DATA_DIR, "metadata.csv")
if os.path.isfile(meta_path):
    meta = pd.read_csv(meta_path)
    print(f"Metadata rows: {len(meta):,}")
    print(f"Columns: {list(meta.columns)}")
    print(f"Label values: {meta['label'].value_counts().to_dict()}")
    meta["label_norm"] = meta["label"].str.lower().str.strip()
    print(f"\nNormalised label counts: {meta['label_norm'].value_counts().to_dict()}")
else:
    print("metadata.csv not found.")

### 3.1.7 Sample images from each class

In [ ]:
def show_samples(df, label, n=8, seed=42):
    """Display a grid of sample images for a given label."""
    subset = df[(df["label"] == label) & df["is_readable"]]
    sample_paths = subset["path"].sample(min(n, len(subset)), random_state=seed).tolist()

    fig, axes = plt.subplots(2, n // 2, figsize=(12, 6))
    axes = axes.flatten()

    for ax, path in zip(axes, sample_paths):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(os.path.splitext(os.path.basename(path))[0], fontsize=8)
        ax.axis("off")

    fig.suptitle(f"Sample — {label}", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

show_samples(df, "fake faces", n=8)
show_samples(df, "real faces", n=8)

### Save inventory

In [ ]:
out_path = os.path.join(REPORTS_DIR, "image_inventory.csv")
df.to_csv(out_path, index=False)
print(f"Inventory saved to {out_path} ({len(df):,} rows)")

### Summary

In [ ]:
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"Total images:        {len(df):,}")
print(f"Readable images:     {df['is_readable'].sum():,}")
print(f"Corrupted images:    {(~df['is_readable']).sum():,}")
print(f"Unique file hashes:  {df['file_hash'].nunique():,}")
print(f"Unique pixel hashes: {df['pixel_hash'].dropna().nunique():,}")
print(f"Image resolutions:   {readable.groupby(['width','height']).ngroups} variant(s)")
print(f"File formats:        {df['extension'].nunique()} type(s)")
print(f"Classes:             {df['label'].nunique()} — {dict(df['label'].value_counts())}")
print("=" * 50)